In [86]:
import pandas as pd
from xgboost import XGBClassifier
import math
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.neighbors import KNeighborsClassifier
from numpy.random import rand
from numpy.random import seed
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from scipy import stats
import re
from functools import reduce
from sklearn.ensemble import RandomForestClassifier
from scipy.optimize import Bounds,  minimize
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
import seaborn as sns
import random
import time
import os
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_extraction.text import CountVectorizer
import requests
#from boruta import BorutaPy
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import ExtraTreesClassifier


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
df_row = pd.read_csv('/content/drive/My Drive/loan.csv')

In [10]:
df_row['loan_status'].value_counts()

,count
loan_status,
Fully Paid,1041952
Current,919695
Charged Off,261655
Late (31-120 days),21897
In Grace Period,8952
Late (16-30 days),3737
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,31


In [11]:
df = df_row.loc[:, df_row.isnull().mean() < .8]
#df = df[(df.loan_status != 'In Grace Period')]
df = df.dropna(thresh=int(len(df.columns)*0.5))
colunas_objetos = df.columns[df.dtypes=='object']
df = df.apply(lambda x: x.fillna(0) if x.dtype.kind in 'biufc' else x.fillna('Sem_Info'))
df.shape

(2257886, 106)

In [12]:
def get_target(target):
    if target == 'Fully Paid':
        return 0
    #elif target == 'Does not meet the credit policy. Status:Fully Paid':
    #    return 0
    elif target == 'Current':
        return 0
    else:
        return 1

In [13]:
df['target'] = df['loan_status'].apply(get_target)
df['target'].value_counts(normalize=True,dropna= False)

,proportion
target,
0,0.867864
1,0.132136


In [14]:
df[colunas_objetos] = df[colunas_objetos].apply(lambda x: x.astype('category').cat.codes)
target = 'target'
safra = 'issue_d'
variaveis = df.columns.drop([target,safra])
correlacoes_p = []
correlacoes_s = []

for COLUNA in variaveis:
  correlacoes_p.append(np.round(df[target].corr(df[COLUNA]),2))
  correlacao, p = spearmanr(df[target], df[COLUNA])
  correlacoes_s.append(np.round(correlacao,2))
df_cor = pd.DataFrame({'Variaveis': variaveis, 'Correlacoes_S': correlacoes_s, 'Correlacoes_P': correlacoes_p})

In [15]:
df_cor = df_cor.dropna()
df_cor['fx']= pd.qcut(df_cor['Correlacoes_P'], q=4)

lista_variaveis_fx = []
for i in list((range(0,len(pd.unique(df_cor['fx']))))):
  lista_variaveis_fx.append(df_cor[df_cor['fx']==pd.unique(df_cor['fx'])[i]].Variaveis)


tamanho_dos_grupos=[]
lista_de_grupos=[]
lista_variaveis_fx[0]
for i in list(range(0,len(lista_variaveis_fx))):
  lista_de_grupos.append('Grupo '+str(i))
  tamanho_dos_grupos.append(len(lista_variaveis_fx[i]))

divisao_grupos = pd.DataFrame({'Grupos':lista_de_grupos,'Tamanho do Grupo':tamanho_dos_grupos})

In [44]:
divisao_grupos

,Grupos,Tamanho do Grupo
0,Grupo 0,13
1,Grupo 1,25
2,Grupo 2,38
3,Grupo 3,28


In [16]:
y = df['target']
X_vetor = []
X_vetor_train = []
X_vetor_test = []
y_vetor_train = []
y_vetor_test = []
for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor.append('X_'+str(i+1))
  X_vetor_train.append('X_train_'+str(i+1))
  X_vetor_test.append('X_test_'+str(i+1))
  y_vetor_train.append('y_train_'+str(i+1))
  y_vetor_test.append('y_test_'+str(i+1))

for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor[i] = df[lista_variaveis_fx[i]]
  X_vetor_train[i], X_vetor_test[i],y_vetor_train[i], y_vetor_test[i]= train_test_split(X_vetor[i], y, test_size = 0.3)

retorno_grupo = []
for i in list(range(0,len(X_vetor))):
  df_x = X_vetor[0]
  df_x[safra]=df[safra]
  df_x[target]=df[target]
  clf = lgb.LGBMClassifier()
  features_name = []
  features_importance = []
  retorno_safra = []
  id_safra = []
  for ANO in pd.unique(df_x[safra]):
    df_xx = df_x[df_x[safra]==ANO]
    x_filtrado = df_xx.drop([safra,target], axis = 1)
    y_filtrado = df_xx.target
    x_filtrado_train, x_filtrado_test, y_filtrado_train, y_filtrado_test= train_test_split(x_filtrado, y_filtrado, test_size = 0.3)
    clf.fit(x_filtrado_train, y_filtrado_train)
    pred_filtrado =clf.predict(x_filtrado_test)
    features_name.append(x_filtrado_train.columns)
    features_importance.append(clf.feature_importances_)
    retorno_safra.append(accuracy_score(pred_filtrado, y_filtrado_test))
    id_safra.append(ANO)
  df_retornos = pd.DataFrame({'Retorno Grupo '+str(i): retorno_safra, 'Safra': id_safra})
  retorno_grupo.append(df_retornos)
df_retornos = reduce(lambda  left,right: pd.merge(left,right,on=['Safra'],how='outer'), retorno_grupo)

A saída de streaming foi truncada nas últimas 5000 linhas.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 57, number of negative: 19
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000044 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 174
[LightGBM] [Info] Number of data points in the train set: 76, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.750000 -> initscore=1.098612
[LightGBM] [Info] Start training from score 1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] 

In [17]:
first_column = df_retornos.pop('Safra')
df_retornos.insert(0, 'Safra', first_column)

In [43]:
importancias_medias = np.round(np.mean(features_importance, axis=0),1)
data = {'Importancias Medias C1': importancias_medias, 'Features C1': features_name[0]}
pd.DataFrame(data).sort_values(by = ['Importancias Medias C1'],ascending = False)

,Importancias Medias C1,Features C1
3,492.5,installment
4,489.0,emp_title
0,287.1,loan_amnt
9,280.2,open_acc
7,240.4,title
5,199.6,emp_length
2,147.8,funded_amnt_inv
11,127.0,mths_since_last_major_derog
6,99.7,purpose
8,46.3,delinq_2yrs


In [52]:
y = df['target']
X_vetor = []
X_vetor_train = []
X_vetor_test = []
y_vetor_train = []
y_vetor_test = []
for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor.append('X_'+str(i+1))
  X_vetor_train.append('X_train_'+str(i+1))
  X_vetor_test.append('X_test_'+str(i+1))
  y_vetor_train.append('y_train_'+str(i+1))
  y_vetor_test.append('y_test_'+str(i+1))

for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor[i] = df[lista_variaveis_fx[i]]
  X_vetor_train[i], X_vetor_test[i],y_vetor_train[i], y_vetor_test[i]= train_test_split(X_vetor[i], y, test_size = 0.3)

retorno_grupo = []
for i in list(range(0,len(X_vetor))):
  df_x = X_vetor[1]
  df_x[safra]=df[safra]
  df_x[target]=df[target]
  clf = lgb.LGBMClassifier()
  features_name = []
  features_importance = []
  retorno_safra = []
  id_safra = []
  for ANO in pd.unique(df_x[safra]):
    df_xx = df_x[df_x[safra]==ANO]
    x_filtrado = df_xx.drop([safra,target], axis = 1)
    y_filtrado = df_xx.target
    x_filtrado_train, x_filtrado_test, y_filtrado_train, y_filtrado_test= train_test_split(x_filtrado, y_filtrado, test_size = 0.3)
    clf.fit(x_filtrado_train, y_filtrado_train)
    pred_filtrado =clf.predict(x_filtrado_test)
    features_name.append(x_filtrado_train.columns)
    features_importance.append(clf.feature_importances_)
    retorno_safra.append(accuracy_score(pred_filtrado, y_filtrado_test))
    id_safra.append(ANO)
  df_retornos = pd.DataFrame({'Retorno Grupo '+str(i): retorno_safra, 'Safra': id_safra})
  retorno_grupo.append(df_retornos)
df_retornos = reduce(lambda  left,right: pd.merge(left,right,on=['Safra'],how='outer'), retorno_grupo)

A saída de streaming foi truncada nas últimas 5000 linhas.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 186, number of negative: 1099
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000244 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1119
[LightGBM] [Info] Number of data points in the train set: 1285, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.144747 -> initscore=-1.776409
[LightGBM] [Info] Start training from score -1.776409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

In [53]:
importancias_medias = np.round(np.mean(features_importance, axis=0),1)
data = {'Importancias Medias C2': importancias_medias, 'Features C2': features_name[0]}
pd.DataFrame(data).sort_values(by = ['Importancias Medias C2'],ascending = False)

,Importancias Medias C2,Features C2
10,271.3,total_rec_int
14,267.2,last_pymnt_d
7,258.4,dti
9,220.4,revol_util
1,199.9,int_rate
16,199.0,last_credit_pull_d
18,148.2,bc_util
17,76.7,acc_open_past_24mths
8,74.9,inq_last_6mths
12,70.2,recoveries


In [54]:
y = df['target']
X_vetor = []
X_vetor_train = []
X_vetor_test = []
y_vetor_train = []
y_vetor_test = []
for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor.append('X_'+str(i+1))
  X_vetor_train.append('X_train_'+str(i+1))
  X_vetor_test.append('X_test_'+str(i+1))
  y_vetor_train.append('y_train_'+str(i+1))
  y_vetor_test.append('y_test_'+str(i+1))

for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor[i] = df[lista_variaveis_fx[i]]
  X_vetor_train[i], X_vetor_test[i],y_vetor_train[i], y_vetor_test[i]= train_test_split(X_vetor[i], y, test_size = 0.3)

retorno_grupo = []
for i in list(range(0,len(X_vetor))):
  df_x = X_vetor[2]
  df_x[safra]=df[safra]
  df_x[target]=df[target]
  clf = lgb.LGBMClassifier()
  features_name = []
  features_importance = []
  retorno_safra = []
  id_safra = []
  for ANO in pd.unique(df_x[safra]):
    df_xx = df_x[df_x[safra]==ANO]
    x_filtrado = df_xx.drop([safra,target], axis = 1)
    y_filtrado = df_xx.target
    x_filtrado_train, x_filtrado_test, y_filtrado_train, y_filtrado_test= train_test_split(x_filtrado, y_filtrado, test_size = 0.3)
    clf.fit(x_filtrado_train, y_filtrado_train)
    pred_filtrado =clf.predict(x_filtrado_test)
    features_name.append(x_filtrado_train.columns)
    features_importance.append(clf.feature_importances_)
    retorno_safra.append(accuracy_score(pred_filtrado, y_filtrado_test))
    id_safra.append(ANO)
  df_retornos = pd.DataFrame({'Retorno Grupo '+str(i): retorno_safra, 'Safra': id_safra})
  retorno_grupo.append(df_retornos)
df_retornos = reduce(lambda  left,right: pd.merge(left,right,on=['Safra'],how='outer'), retorno_grupo)

A saída de streaming foi truncada nas últimas 5000 linhas.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

In [55]:
importancias_medias = np.round(np.mean(features_importance, axis=0),1)
data = {'Importancias Medias C3': importancias_medias, 'Features C3': features_name[0]}
pd.DataFrame(data).sort_values(by = ['Importancias Medias C3'],ascending = False)

,Importancias Medias C3,Features C3
5,277.5,revol_bal
3,269.6,earliest_cr_line
0,264.2,annual_inc
1,248.8,zip_code
6,173.2,total_acc
2,126.2,addr_state
20,122.5,mo_sin_old_il_acct
36,117.8,total_bal_ex_mort
4,107.9,mths_since_last_delinq
37,91.4,total_il_high_credit_limit


In [56]:
y = df['target']
X_vetor = []
X_vetor_train = []
X_vetor_test = []
y_vetor_train = []
y_vetor_test = []
for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor.append('X_'+str(i+1))
  X_vetor_train.append('X_train_'+str(i+1))
  X_vetor_test.append('X_test_'+str(i+1))
  y_vetor_train.append('y_train_'+str(i+1))
  y_vetor_test.append('y_test_'+str(i+1))

for i in list(range(0,len(lista_variaveis_fx))):
  X_vetor[i] = df[lista_variaveis_fx[i]]
  X_vetor_train[i], X_vetor_test[i],y_vetor_train[i], y_vetor_test[i]= train_test_split(X_vetor[i], y, test_size = 0.3)

retorno_grupo = []
for i in list(range(0,len(X_vetor))):
  df_x = X_vetor[3]
  df_x[safra]=df[safra]
  df_x[target]=df[target]
  clf = lgb.LGBMClassifier()
  features_name = []
  features_importance = []
  retorno_safra = []
  id_safra = []
  for ANO in pd.unique(df_x[safra]):
    df_xx = df_x[df_x[safra]==ANO]
    x_filtrado = df_xx.drop([safra,target], axis = 1)
    y_filtrado = df_xx.target
    x_filtrado_train, x_filtrado_test, y_filtrado_train, y_filtrado_test= train_test_split(x_filtrado, y_filtrado, test_size = 0.3)
    clf.fit(x_filtrado_train, y_filtrado_train)
    pred_filtrado =clf.predict(x_filtrado_test)
    features_name.append(x_filtrado_train.columns)
    features_importance.append(clf.feature_importances_)
    retorno_safra.append(accuracy_score(pred_filtrado, y_filtrado_test))
    id_safra.append(ANO)
  df_retornos = pd.DataFrame({'Retorno Grupo '+str(i): retorno_safra, 'Safra': id_safra})
  retorno_grupo.append(df_retornos)
df_retornos = reduce(lambda  left,right: pd.merge(left,right,on=['Safra'],how='outer'), retorno_grupo)

A saída de streaming foi truncada nas últimas 5000 linhas.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

In [57]:
importancias_medias = np.round(np.mean(features_importance, axis=0),1)
data = {'Importancias Medias C4': importancias_medias, 'Features C4': features_name[0]}
pd.DataFrame(data).sort_values(by = ['Importancias Medias C4'],ascending = False)

,Importancias Medias C4,Features C4
4,181.4,total_pymnt
0,162.5,loan_status
2,118.2,out_prncp
6,29.8,total_rec_prncp
7,27.8,last_pymnt_amnt
9,19.7,tot_cur_bal
5,17.5,total_pymnt_inv
19,16.0,mo_sin_old_rev_tl_op
12,15.8,total_bal_il
11,14.9,mths_since_rcnt_il


## Final Dataset Features

In [60]:
divisao_grupos['Alocacao Otima'] = [0.26, 0.00, 0.68, 0.06]
divisao_grupos['Tamanho Final do Grupo'] = round((divisao_grupos['Tamanho do Grupo']*divisao_grupos['Alocacao Otima'])/divisao_grupos['Alocacao Otima'].sum(),0)
divisao_grupos

,Grupos,Tamanho do Grupo,Alocacao Otima,Tamanho Final do Grupo
0,Grupo 0,13,0.26,3.0
1,Grupo 1,25,0.00,0.0
2,Grupo 2,38,0.68,26.0
3,Grupo 3,28,0.06,2.0


In [70]:
Features_G0 = ['installment','emp_title','loan_amnt']
Features_G1 = []
Features_G2 = ['revol_bal','earliest_cr_line','annual_inc','zip_code','total_acc','addr_state','mo_sin_old_il_acct','total_bal_ex_mort','mths_since_last_delinq','total_il_high_credit_limit','pct_tl_nvr_dlq'
,'num_rev_accts','num_il_tl','num_sats','num_bc_tl','num_op_rev_tl','num_actv_bc_tl','mths_since_recent_revol_delinq','num_bc_sats','mths_since_recent_bc_dlq'
,'tot_coll_amt','open_rv_24m','inq_last_12m','open_il_24m','total_cu_tl','num_accts_ever_120_pd']
Features_G3 = ['total_pymnt','loan_status']

In [71]:
FEATURES_FINAL = Features_G0 +Features_G1+Features_G2+Features_G3

In [72]:
len(FEATURES_FINAL)

31

## Modelo Final

In [74]:
param_grid = {
"num_leaves": [31, 63, 127],
"max_depth": [-1, 3, 9],
"subsample": [0.4, 1.0],
"colsample_bytree": [0.7, 1.0]
}


X_train_final, X_test_final,y_train_final, y_test_final= train_test_split(df[FEATURES_FINAL], df[target], test_size = 0.3)
scaler = MinMaxScaler()
X_train_final = scaler.fit_transform(X_train_final)
X_test_final = scaler.transform(X_test_final)


clf = lgb.LGBMClassifier(objective="binary", metric="auc", random_state=42)
grid = GridSearchCV(clf, param_grid, cv=10, scoring="accuracy")
grid.fit(X_train_final, y_train_final)
predicao = grid.predict(X_test_final)


acuracia_CSM_model = accuracy_score(predicao, y_test_final)

A saída de streaming foi truncada nas últimas 5000 linhas.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 187

In [87]:
acuracia = np.round(acuracia_CSM_model,4)*100
f1_score = (np.round(f1_score(predicao, y_test_final,average='weighted'),5)*100)
precisao = (np.round(precision_score(predicao, y_test_final, average='weighted'),5)*100)
recall = (np.round(recall_score(predicao, y_test_final,average='macro'),4)*100)

In [101]:
Performance_Teste = pd.DataFrame({'Accuracy':[96.8,acuracia],
                       'F1 Score': [96.44,f1_score],
                       'Precision':[99.7,precisao],
                       'Recall':[93.4,recall],
                                  'Selection Technique': ['Random', 'Hierarchical']})


In [102]:
Performance_Teste = Performance_Teste.set_index('Selection Technique')
Performance_Teste

,Accuracy,F1 Score,Precision,Recall
Selection Technique,,,,
Random,96.8,96.44,99.70,93.40
Hierarchical,96.8,96.44,99.77,93.32
